<a href="https://colab.research.google.com/github/huggingface/deep-rl-class/blob/main/notebooks/unit6/unit6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第 6 单元:基于 Panda-Gym 机器人仿真的优势 Actor-Critic(A2C)🤖

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit8/thumbnail.png"  alt="Thumbnail"/>

在本 notebook 中,你将学习如何在 [Panda-Gym](https://github.com/qgallouedec/panda-gym) 中使用 A2C。你将**训练一只机械臂**(Franka Emika Panda 机器人)来完成一项任务:

- `Reach`(够取):机器人必须把它的末端执行器(end-effector)放到目标位置上。

完成之后,你就可以**在其他机器人任务中进行训练**了。


### 🎮 环境:

- [Panda-Gym](https://github.com/qgallouedec/panda-gym)

###📚 强化学习库:

- [Stable-Baselines3](https://stable-baselines3.readthedocs.io/)

我们一直在努力改进教程,所以**如果你在本 notebook 中发现了一些问题**,请[在 GitHub 仓库上提交 issue](https://github.com/huggingface/deep-rl-class/issues)。

## 本 notebook 的目标 🏆

完成本 notebook 后,你将能够:

- 能够使用环境库 **Panda-Gym**。
- 能够**使用 A2C 训练机器人**。
- 理解**为什么我们需要对输入进行归一化**。
- 能够**把训练好的智能体和代码推送到 Hub**,并附上精彩的回放视频和评估分数 🔥。




## 本 notebook 来自深度强化学习课程
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Deep RL Course illustration"/>

在这个免费课程中,你将:

- 📖 从**理论与实践**两方面学习深度强化学习。
- 🧑‍💻 学会**使用著名的深度强化学习库**,例如 Stable Baselines3、RL Baselines3 Zoo、CleanRL 和 Sample Factory 2.0。
- 🤖 在**独特的环境中训练智能体**

更多内容请查看 📚 课程大纲 👉 https://simoninithomas.github.io/deep-rl-course

别忘了**<a href="http://eepurl.com/ic5ZUD">注册本课程</a>**(我们收集你的邮箱,是为了**在每个单元发布时把链接发送给你,并向你提供挑战和更新的相关信息)。**


与我们保持联系的最佳方式是加入我们的 Discord 服务器,与社区以及我们交流 👉🏻 https://discord.gg/ydHrjt3WP5

## 前置要求 🏗️
在深入学习本 notebook 之前,你需要:

🔲 📚 阅读第 6 单元,学习 [Actor-Critic 方法](https://huggingface.co/deep-rl-course/unit6/introduction) 🤗  

# 让我们来训练第一批机器人 🤖

要通过[认证流程](https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process)的验证,你需要把训练好的模型推送到 Hub,并取得以下结果:

- `PandaReachDense-v3`(Panda 机械臂够取任务的稠密奖励版本)需取得 >= -3.5 的结果。

要查看你的结果,请前往[排行榜](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard)找到你的模型,**结果 = 平均奖励(mean_reward)- 奖励的标准差(std of reward)**

有关认证流程的更多信息,请查看这一节 👉 https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process

## 设置 GPU 💪
- 为了**加速智能体的训练,我们将使用 GPU**。为此,请前往 `Runtime > Change Runtime type`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step1.jpg" alt="GPU Step 1">

- `Hardware Accelerator > GPU`(硬件加速器 > GPU)

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step2.jpg" alt="GPU Step 2">

## 创建虚拟显示器 🔽

在本 notebook 中,我们需要生成一段回放视频。为此,在 Colab 上**我们需要有一个虚拟屏幕才能渲染环境**(从而录制画面帧)。

因此,下面的单元格将安装相关库,并创建并启动一个虚拟屏幕 🖥

In [ ]:
%%capture
!apt install python-opengl
!apt install ffmpeg
!apt install xvfb
!pip3 install pyvirtualdisplay

In [ ]:
# 虚拟显示器
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

### 安装依赖 🔽

第一步是安装依赖,我们将安装以下几个库:
- `gymnasium`
- `panda-gym`:包含机械臂环境。
- `stable-baselines3`:SB3 深度强化学习库。
- `huggingface_sb3`:为 Stable-Baselines3 提供的附加代码,用于从 Hugging Face 🤗 Hub 加载和上传模型。
- `huggingface_hub`:让任何人都能操作 Hub 仓库的库。

⏲ 安装过程可能**需要 10 分钟**。

In [ ]:
!pip install stable-baselines3[extra]
!pip install gymnasium

In [ ]:
!pip install huggingface_sb3
!pip install huggingface_hub
!pip install panda_gym

## 导入软件包 📦

In [ ]:
import os

import gymnasium as gym
import panda_gym

from huggingface_sb3 import load_from_hub, package_to_hub

from stable_baselines3 import A2C
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.env_util import make_vec_env

from huggingface_hub import notebook_login

## PandaReachDense-v3 🦾

我们要训练的智能体是一只机械臂,它需要执行控制任务(移动机械臂并使用末端执行器)。

在机器人学中,*末端执行器(end-effector)*是安装在机械臂末端、用于与环境交互的装置。

在 `PandaReach`(够取)环境中,机器人必须把它的末端执行器放到目标位置(绿球)上。

我们将使用该环境的稠密(dense)版本。这意味着我们会得到一个*稠密奖励函数(dense reward function)*,它**会在每个时间步都给出奖励**(智能体越接近完成任务,奖励越高)。与之相对的是*稀疏奖励函数(sparse reward function)*,那种情况下环境**当且仅当任务完成时才返回奖励**。

此外,我们将使用*末端执行器位移控制(End-effector displacement control)*,也就是说**动作对应的是末端执行器的位移**。我们不单独控制每个关节的运动(即关节控制)。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit8/robotics.jpg"  alt="Robotics"/>


这样**训练会更容易**。



### 创建环境

#### 环境 🎮

在 `PandaReachDense-v3` 中,机械臂必须把它的末端执行器放到目标位置(绿球)上。

In [ ]:
env_id = "PandaReachDense-v3"

# 创建环境
env = gym.make(env_id)

# 获取状态空间与动作空间
s_size = env.observation_space.shape
a_size = env.action_space

In [ ]:
print("_____OBSERVATION SPACE_____ \n")
print("The State Space is: ", s_size)
print("Sample observation", env.observation_space.sample()) # 获取一个随机观测

观测空间**是一个包含 3 个不同元素的字典**:
- `achieved_goal`: (x,y,z) 末端执行器的当前位置。
- `desired_goal`: (x,y,z) 末端执行器的目标位置。
- `observation`: 末端执行器的位置 (x,y,z) 与速度 (vx, vy, vz)。

由于观测是一个字典,**我们需要使用 MultiInputPolicy 策略,而不是 MlpPolicy**。

In [ ]:
print("\n _____ACTION SPACE_____ \n")
print("The Action Space is: ", a_size)
print("Action Space Sample", env.action_space.sample()) # 随机执行一个动作

动作空间是一个包含 3 个值的向量:
- 控制 x、y、z 方向上的移动

### 对观测和奖励进行归一化

在强化学习中,一个良好实践是[对输入特征进行归一化](https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html)。

为此,有一个包装器(wrapper)会计算输入特征的滑动平均值和标准差。

我们也可以用同一个包装器来归一化奖励,只需加上 `norm_reward = True`

[你应该查阅文档来填写这个单元格](https://stable-baselines3.readthedocs.io/en/master/guide/vec_envs.html#vecnormalize)

In [ ]:
env = make_vec_env(env_id, n_envs=4)

# 添加这个包装器,以对观测和奖励进行归一化
env = # TODO: 添加这个包装器

#### 解答

In [ ]:
env = make_vec_env(env_id, n_envs=4)

env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)

### 创建 A2C 模型 🤖

有关 Stable-Baselines3 中 A2C 实现的更多信息,请查看:https://stable-baselines3.readthedocs.io/en/master/modules/a2c.html#notes

为了找到最佳参数,我参考了 [Stable-Baselines3 团队官方训练的智能体](https://huggingface.co/sb3)。

In [ ]:
model = # 创建 A2C 模型,并尝试找到最佳参数

#### 解答

In [ ]:
model = A2C(policy = "MultiInputPolicy",
            env = env,
            verbose=1)

### 训练 A2C 智能体 🏃
- 我们将把智能体训练 1,000,000 个时间步,别忘了在 Colab 上使用 GPU。大约需要 25-40 分钟

In [ ]:
model.learn(1_000_000)

In [ ]:
# 保存模型,并在保存智能体时一并保存 VecNormalize 统计量
model.save("a2c-PandaReachDense-v3")
env.save("vec_normalize.pkl")

### 评估智能体 📈
- 现在我们的智能体已经训练完成,我们需要**检验它的性能**。
- Stable-Baselines3 提供了一个方法来完成这件事:`evaluate_policy`

In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# 加载已保存的统计量
eval_env = DummyVecEnv([lambda: gym.make("PandaReachDense-v3")])
eval_env = VecNormalize.load("vec_normalize.pkl", eval_env)

# 我们需要覆盖 render_mode
eval_env.render_mode = "rgb_array"

# 测试时不要更新它们
eval_env.training = False
# 测试时不需要对奖励进行归一化
eval_env.norm_reward = False

# 加载智能体
model = A2C.load("a2c-PandaReachDense-v3")

mean_reward, std_reward = evaluate_policy(model, eval_env)

print(f"Mean reward = {mean_reward:.2f} +/- {std_reward:.2f}")

### 把你训练好的模型发布到 Hub 🔥
既然我们已经看到训练取得了不错的结果,就可以用一行代码把训练好的模型发布到 Hub。

📚 相关库的文档 👉 https://github.com/huggingface/huggingface_sb3/tree/main#hugging-face--x-stable-baselines3-v20


通过使用 `package_to_hub`,正如我们在前面单元中已经提到的,**你可以评估智能体、录制回放、为你的智能体生成模型卡(model card)并把它推送到 Hub**。

这样一来:
- 你可以**展示自己的成果** 🔥
- 你可以**观看你的智能体实际游玩** 👀
- 你可以**与社区分享一个他人也能使用的智能体** 💾
- 你可以**访问排行榜 🏆,看看你的智能体与同学们相比表现如何** 👉 https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard


要与社区分享你的模型,还需要完成以下三个步骤:

1️⃣ (如果还没有的话)注册一个 HF 账号 ➡ https://huggingface.co/join

2️⃣ 登录之后,你需要保存来自 Hugging Face 网站的认证令牌(token)。
- 创建一个新令牌(https://huggingface.co/settings/tokens),**角色需选择 write(写权限)**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/create-token.jpg" alt="Create HF Token">

- 复制该令牌
- 运行下面的单元格并粘贴令牌

In [ ]:
notebook_login()
!git config --global credential.helper store

如果你不想使用 Google Colab 或 Jupyter Notebook,就需要改用这个命令:`huggingface-cli login`

3️⃣ 现在我们可以使用 `package_to_hub()` 函数,把训练好的智能体推送到 🤗 Hub 🔥 了

对于这个环境,**运行这个单元格大约需要 10 分钟**

In [ ]:
from huggingface_sb3 import package_to_hub

package_to_hub(
    model=model,
    model_name=f"a2c-{env_id}",
    model_architecture="A2C",
    env_id=env_id,
    eval_env=eval_env,
    repo_id=f"ThomasSimonini/a2c-{env_id}", # 修改为你的用户名
    commit_message="Initial commit",
)

## 更多附加挑战 🏆
学习的最佳方式**就是亲自动手尝试**!为什么不试试 `PandaPickAndPlace-v3`(拾取与放置)呢?

如果你想尝试 panda-gym 中更高级的任务,你需要看看使用 **TQC 或 SAC**(对样本效率要求更高、适合机器人任务的算法)都做了哪些工作。在真实的机器人系统中,你会使用样本效率更高的算法,原因很简单:与仿真不同,**如果让机械臂过度运动,你会有损坏它的风险**。

PandaPickAndPlace-v1(该模型使用的是环境的 v1 版本):https://huggingface.co/sb3/tqc-PandaPickAndPlace-v1

也请务必查看 panda-gym 的文档:https://panda-gym.readthedocs.io/en/latest/usage/train_with_sb3.html

我们为你提供了训练另一个智能体的步骤(可选):

1. 定义名为 "PandaPickAndPlace-v3" 的环境
2. 创建向量化环境
3. 添加一个包装器来归一化观测和奖励。[查阅文档](https://stable-baselines3.readthedocs.io/en/master/guide/vec_envs.html#vecnormalize)
4. 创建 A2C 模型(别忘了设置 verbose=1 以打印训练日志)。
5. 训练 1M(100 万)个时间步
6. 保存模型,并在保存智能体时一并保存 VecNormalize 统计量
7. 评估你的智能体
8. 使用 `package_to_hub` 把你训练好的模型发布到 Hub 🔥


### 解答(可选)

In [ ]:
# 1 - 2
env_id = "PandaPickAndPlace-v3"
env = make_vec_env(env_id, n_envs=4)

# 3
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)

# 4
model = A2C(policy = "MultiInputPolicy",
            env = env,
            verbose=1)
# 5
model.learn(1_000_000)

In [ ]:
# 6
model_name = "a2c-PandaPickAndPlace-v3";
model.save(model_name)
env.save("vec_normalize.pkl")

# 7
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# 加载已保存的统计量
eval_env = DummyVecEnv([lambda: gym.make("PandaPickAndPlace-v3")])
eval_env = VecNormalize.load("vec_normalize.pkl", eval_env)

# 测试时不要更新它们
eval_env.training = False
# 测试时不需要对奖励进行归一化
eval_env.norm_reward = False

# 加载智能体
model = A2C.load(model_name)

mean_reward, std_reward = evaluate_policy(model, eval_env)

print(f"Mean reward = {mean_reward:.2f} +/- {std_reward:.2f}")

# 8
package_to_hub(
    model=model,
    model_name=f"a2c-{env_id}",
    model_architecture="A2C",
    env_id=env_id,
    eval_env=eval_env,
    repo_id=f"ThomasSimonini/a2c-{env_id}", # TODO: 修改为你的用户名
    commit_message="Initial commit",
)

第 7 单元再见!🔥
## 持续学习,保持出色 🤗